# 🚀 CommunityLab — Laboratorio Interactivo de IA (Data Science)
**Equipo 39 · ONE G10 LATAM (Oracle Next Education & Alura)**  
**Data Scientist:** Danny de Jesús González Portillo  
**Línea Base:** Semana 3 consolidada (`src.agentes.procesamiento`, `src.agentes.grafo`, `src.datos.ingesta`)

---

### 🎯 Propósito de este Notebook:
Este cuaderno interactivo te permite testear de punta a punta el pipeline de **Semana 3** de forma visual, controlada y modular:

1. **Capa de Ingesta y Relevancia (`src.datos.ingesta` y `src.datos.relevancia`):** Ver cómo se limpian los datos simulados y cómo se calcula el score matemático de relevancia para separar la población general del grupo elegible para marketing.
2. **Capa del Grafo de Agentes (`src.agentes.grafo`):** Ejecutar los estados de las interacciones a través del grafo con sus nodos de análisis cognitivo (`nodo_analizador`), enrutamiento (`nodo_enrutador`) y generación de activos.
3. **Capa de Procesamiento Completo de Entrega (`src.agentes.procesamiento`):** Procesar el paquete completo por ciclos planificados con observabilidad y reintentos automáticos.
4. **Inspección de Prompts Few-Shot y Salidas Estructuradas:** Validar el contrato JSON de salida de activos (LinkedIn, Boletín, FAQs, Casos de éxito e Insights de mejora).

In [9]:
import os
import sys
import json
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv

# 1. Cargar variables de entorno locales (.env)
load_dotenv(override=True)

# 2. Asegurar que la raíz del proyecto esté en sys.path
RAIZ_PROYECTO = Path(".").resolve()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from src.agentes.configuracion_ia import (
    obtener_proveedor_analisis,
    obtener_proveedor_generacion,
    MODELO_NVIDIA_NEMOTRON,
)
from src.agentes.modelo_ia import obtener_configuracion_modelo

nvidia_key = os.getenv("NVIDIA_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")
cohere_key = os.getenv("COHERE_API_KEY")

config_activa = obtener_configuracion_modelo()
prov_analisis = obtener_proveedor_analisis().upper()
prov_gen = obtener_proveedor_generacion().upper()

print("=" * 75)
print("✅ Entorno de CommunityLab Inicializado Correctamente (Semana 3 Consolidada)")
print("=" * 75)
print(f"📁 Raíz del proyecto      : {RAIZ_PROYECTO}")
print(f"⚡ Proveedor Análisis      : {prov_analisis} (NVIDIA NIM por defecto)")
print(f"🎨 Proveedor Generación    : {prov_gen}")
print(f"🧠 Modelo Activo Primario  : {config_activa.get('identificador_modelo')}")
print("🛡️ Fallback Automático     : Gemini (Activo ante 429, timeouts o fallos)")
print(f"🔑 NVIDIA_API_KEY          : {'✅ Configurada' if nvidia_key else '⚠️ Pendiente'}")
print(f"🔑 GEMINI_API_KEY          : {'✅ Configurada' if gemini_key else '⚠️ Pendiente'}")
print(f"🔑 COHERE_API_KEY          : {'✅ Configurada' if cohere_key else '⚠️ Pendiente'}")
print("=" * 75)


✅ Entorno de CommunityLab Inicializado Correctamente (Semana 3 Consolidada)
📁 Raíz del proyecto      : C:\Users\Danny\Documents\Proyecto de CommunityLab MotorInteligente\communitylab
⚡ Proveedor Análisis      : NVIDIA_NIM (NVIDIA NIM por defecto)
🎨 Proveedor Generación    : NVIDIA_NIM
🧠 Modelo Activo Primario  : nvidia/nemotron-3.5-lightning-30b-a3b
🛡️ Fallback Automático     : Gemini (Activo ante 429, timeouts o fallos)
🔑 NVIDIA_API_KEY          : ✅ Configurada
🔑 GEMINI_API_KEY          : ✅ Configurada
🔑 COHERE_API_KEY          : ✅ Configurada


---
## 📦 Paso 1: Ingesta de Datos y Scoring de Relevancia (`src.datos.ingesta`)
Probamos la carga del dataset maestro (`src/datos/mensajes_comunidad_simulados.json`), la limpieza y el filtro de relevancia.
El pipeline clasifica cada interacción en:
- **`incluido_sentimiento`**: Todo mensaje con contenido útil se incluye en el análisis global de salud de la comunidad.
- **`elegible_contenido`**: Solo mensajes con score >= 70 y contenido sustancial son elegibles para transformarse en activos de marketing.

In [10]:
from scripts.apoyo_demostraciones import cargar_paquete_demostracion, obtener_evaluaciones_por_id
from src.datos.ingesta import limpiar_texto

# 1. Cargar el paquete oficial preparado por Datos para el flujo de IA
paquete = cargar_paquete_demostracion(tamano_ciclo=12)
evaluaciones = obtener_evaluaciones_por_id(paquete['informe'])

resumen = paquete['informe']['resumen']
total_interacciones = resumen.get('total', len(paquete['estados']))
seleccionadas = resumen.get('seleccionadas')
elegibles = len(paquete['plan']['ids_contenido'])
ciclos = len(paquete['plan']['ciclos'])

print(f"📊 Total de interacciones procesadas: {total_interacciones}")
print(f"✅ Interacciones seleccionadas: {seleccionadas}")
print(f"🎯 Mensajes elegibles para activos de marketing: {elegibles}")
print(f"🔄 Ciclos planificados por lotes: {ciclos}")

# Mostrar una muestra de los mensajes evaluados
print("\n" + "-" * 70)
print("Muestra de Mensajes y sus Puntuaciones de Relevancia (DA):")
print("-" * 70)
for estado in paquete['estados'][:4]:
    msg_id = estado['id']
    autor = estado.get('autor', 'N/A')
    canal = estado.get('canal', 'N/A')
    score = estado.get('score_relevancia')
    tipo = estado.get('tipo_original')
    texto = estado.get('texto', '')[:85]
    elegible = '✅ SÍ' if msg_id in paquete['plan']['ids_contenido'] else '❌ NO'
    print(f"🔹 ID: {msg_id} | Autor: {autor} | Canal: {canal}")
    print(f"   Score: {score}/100 | Elegible Activos: {elegible} | Tipo: {tipo}")
    print(f"   Texto limpio: \"{texto}...\"\n")

📊 Total de interacciones procesadas: 23
✅ Interacciones seleccionadas: 19
🎯 Mensajes elegibles para activos de marketing: 19
🔄 Ciclos planificados por lotes: 2

----------------------------------------------------------------------
Muestra de Mensajes y sus Puntuaciones de Relevancia (DA):
----------------------------------------------------------------------
🔹 ID: int-001 | Autor: Camila Restrepo | Canal: #logros-y-empleos
   Score: 90/100 | Elegible Activos: ✅ SÍ | Tipo: testimonio
   Texto limpio: "Comunidad, gracias a este programa conseguí mi primer trabajo como analista de datos ..."

🔹 ID: int-002 | Autor: Diego Fernández | Canal: #dudas-langgraph
   Score: 80/100 | Elegible Activos: ✅ SÍ | Tipo: pregunta_tecnica
   Texto limpio: "Tengo dudas sobre cómo estructurar los nodos condicionales en LangGraph cuando la res..."

🔹 ID: int-003 | Autor: Andrés Kuri | Canal: #logros-y-empleos
   Score: 75/100 | Elegible Activos: ✅ SÍ | Tipo: testimonio
   Texto limpio: "Pasé de no saber pro

---
## 🧠 Paso 2: El Grafo de LangGraph y Enrutamiento (`src.agentes.grafo`)
Probamos la estructura de los nodos cognitivos y el enrutador condicional.
En este paso vemos cómo el estado fluye hacia:
- `linkedin`: Para testimonios de alta relevancia y logros profesionales.
- `preguntas_frecuentes`: Para dudas técnicas recurrentes.
- `boletin`: Para novedades y anuncios del programa.
- `caso_exito`: Para historias de éxito con alto impacto.
- `insight_mejora`: Para retroalimentación y sugerencias críticas.

In [11]:
from src.agentes.modelos import AnalisisMensaje
from src.agentes.estado_agente import EstadoAgente
from src.agentes.nodos.nodo_enrutador import determinar_rutas

# 1. Definimos dos estados simulados con su análisis para probar el enrutador
estado_testimonio: EstadoAgente = {
    'id': 'int-022',
    'texto': 'Comunidad, quedé seleccionada como Desarrolladora Junior de IA!',
    'sentimiento': 'muy_positivo',
    'tema_principal': 'empleabilidad',
    'subtema': 'primer_empleo',
    'tipo_detectado': 'testimonio',
    'elegible_contenido': True,
    'score_relevancia': 95,
}

estado_duda_tecnica: EstadoAgente = {
    'id': 'int-002',
    'texto': 'Tengo dudas sobre cómo estructurar los nodos condicionales en LangGraph.',
    'sentimiento': 'neutral',
    'tema_principal': 'datos_ia',
    'subtema': 'langgraph',
    'tipo_detectado': 'pregunta_tecnica',
    'elegible_contenido': True,
    'score_relevancia': 80,
}

# 2. Evaluar las rutas decididas por el enrutador para cada caso
print('🎯 EVALUACIÓN DE ENRUTAMIENTO CONDICIONAL (LangGraph Router):')
print('-' * 70)
for caso in [estado_testimonio, estado_duda_tecnica]:
    res = determinar_rutas(caso)
    print(f"📌 Caso [{caso['id']}] - Tipo: '{caso['tipo_detectado']}' | Sentimiento: '{caso['sentimiento']}'")
    print(f"   👉 Rutas asignadas por el grafo: {res['rutas']}\n")

🎯 EVALUACIÓN DE ENRUTAMIENTO CONDICIONAL (LangGraph Router):
----------------------------------------------------------------------
📌 Caso [int-022] - Tipo: 'testimonio' | Sentimiento: 'muy_positivo'
   👉 Rutas asignadas por el grafo: ['caso_exito', 'boletin', 'linkedin']

📌 Caso [int-002] - Tipo: 'pregunta_tecnica' | Sentimiento: 'neutral'
   👉 Rutas asignadas por el grafo: ['preguntas_frecuentes']



---
## ⚙️ Paso 3: Procesamiento Completo del Paquete (`src.agentes.procesamiento`)
Probamos la función principal de entrega: `procesar_paquete_entrega` y `procesar_estados_por_lotes`.
Esta arquitectura:
1. Valida los estados de entrada y el plan de ciclos planificados.
2. Ejecuta el procesamiento por lotes (`procesar_estados_por_lotes`).
3. Registra eventos de observabilidad (tiempos, duraciones en milisegundos).
4. Aplica pasadas de recuperación automática sobre fallos transitorios.

---
## 🔬 Paso 3.1: Prueba en Vivo con LLM (Casos de Empleo y Dudas Técnicas)
Probamos la cadena cognitiva en tiempo real utilizando las plantillas de producción de `src/agentes/prompts/analisis.py`.
El motor procesa el texto, extrae el esquema estructurado `AnalisisMensaje` y lo envía al enrutador para decidir los activos a generar.

In [12]:
from src.agentes.cadenas import cadena_analisis
from src.agentes.modelos import AnalisisMensaje
from src.agentes.nodos.nodo_enrutador import determinar_rutas
from src.agentes.estado_agente import EstadoAgente
from src.agentes.observabilidad import config_ejecucion

# 1. Definir dos casos reales del proyecto (Testimonio de Empleo y Duda Técnica)
casos_prueba = [
    {
        "id": "int-022",
        "autor": "Mariana Souza",
        "canal": "#logros-y-empleos",
        "origen": "Discord",
        "tipo_original": "testimonio",
        "score_relevancia": 95,
        "elegible_contenido": True,
        "texto": "Comunidad, gracias al curso y al proyecto de IA de mi portfolio quedé seleccionada como Desarrolladora Junior de IA! Muy agradecida!"
    },
    {
        "id": "int-002",
        "autor": "Diego Fernández",
        "canal": "#dudas-langgraph",
        "origen": "Discord",
        "tipo_original": "pregunta_tecnica",
        "score_relevancia": 80,
        "elegible_contenido": True,
        "texto": "Tengo dudas sobre cómo estructurar los nodos condicionales en LangGraph cuando la respuesta del LLM necesita reintento. ¿Alguien tiene un ejemplo práctico?"
    }
]

print("🚀 PROCESANDO CASOS CON LA CADENA OFICIAL DE ANÁLISIS (NVIDIA NIM + FALLBACK GEMINI):")
print("=" * 75)
for caso in casos_prueba:
    print(f"\n📩 Analizando caso [{caso['id']}] de '{caso['autor']}'...")
    
    # Invocamos la cadena oficial pasando intento=1 requerido por config_ejecucion
    config_obs = config_ejecucion("analisis_demostracion", intento=1, id_interaccion=caso["id"])
    analisis: AnalisisMensaje = cadena_analisis.invoke({
        "canal": caso["canal"],
        "idioma": "es",
        "origen": caso["origen"],
        "tipo_original": caso["tipo_original"],
        "texto": caso["texto"]
    }, config=config_obs)
    
    print(f"   ✅ Sentimiento     : {analisis.sentimiento}")
    print(f"   ✅ Tema principal  : {analisis.tema_principal} ({analisis.subtema})")
    print(f"   ✅ Tipo detectado  : {analisis.tipo_detectado}")
    
    # Construimos el estado para el enrutador determinista
    estado_evaluado: EstadoAgente = {
        "id": caso["id"],
        "texto": caso["texto"],
        "sentimiento": analisis.sentimiento,
        "tema_principal": analisis.tema_principal,
        "subtema": analisis.subtema,
        "tipo_detectado": analisis.tipo_detectado,
        "elegible_contenido": caso["elegible_contenido"],
        "rutas": [],
        "activos_generados": {}
    }
    
    rutas = determinar_rutas(estado_evaluado)
    print(f"   🔀 Rutas asignadas : {rutas}")

print("=" * 75)
print("🎉 ¡Análisis y enrutamiento en vivo completados con éxito!")


🚀 PROCESANDO CASOS CON LA CADENA OFICIAL DE ANÁLISIS (NVIDIA NIM + FALLBACK GEMINI):

📩 Analizando caso [int-022] de 'Mariana Souza'...
   ✅ Sentimiento     : muy_positivo
   ✅ Tema principal  : empleabilidad (seleccionada como Desarrolladora Junior de IA)
   ✅ Tipo detectado  : testimonio
   🔀 Rutas asignadas : {'rutas': ['caso_exito', 'boletin', 'linkedin']}

📩 Analizando caso [int-002] de 'Diego Fernández'...
   ✅ Sentimiento     : neutral
   ✅ Tema principal  : datos_ia (nodos condicionales LangGraph)
   ✅ Tipo detectado  : pregunta_tecnica
   🔀 Rutas asignadas : {'rutas': ['preguntas_frecuentes']}
🎉 ¡Análisis y enrutamiento en vivo completados con éxito!


In [13]:
from scripts.apoyo_demostraciones import cargar_paquete_demostracion
from src.agentes.grafo import procesar_estados_por_lotes

# 1. Cargar el paquete completo de demostración preparado por Datos
paquete_demostracion = cargar_paquete_demostracion(tamano_ciclo=12)

print(f"📦 Paquete cargado:")
print(f"   • Total estados: {len(paquete_demostracion['estados'])}")
print(f"   • Mensajes en plan de contenido: {len(paquete_demostracion['plan']['ids_contenido'])}")
print(f"   • Ciclos planificados: {len(paquete_demostracion['plan']['ciclos'])}")

# 2. Probamos una corrida controlada sobre 2 estados seleccionados
ids_seleccionados = ["int-022", "int-002"]
estados_muestra = [
    e for e in paquete_demostracion["estados"] 
    if e["id"] in ids_seleccionados
]
ids_contenido = set(paquete_demostracion["plan"]["ids_contenido"])

print("\n🚀 Ejecutando `procesar_estados_por_lotes` sobre los estados seleccionados...")
resultados_lote = procesar_estados_por_lotes(estados_muestra, ids_contenido=ids_contenido)

print(f"✅ Procesamiento finalizado. Total resultados obtenidos: {len(resultados_lote)}")
for res in resultados_lote:
    print(f"\n🔹 Resultado ID: {res['id']}")
    print(f"   Sentimiento: {res.get('sentimiento')} | Tema: {res.get('tema_principal')}")
    print(f"   Rutas activadas: {res.get('rutas')}")
    print(f"   Activos generados: {list(res.get('activos_generados', {}).keys())}")
    if res.get('errores'):
        print(f"   ⚠️ Nota/Errores: {res['errores']}")

📦 Paquete cargado:
   • Total estados: 23
   • Mensajes en plan de contenido: 19
   • Ciclos planificados: 2

🚀 Ejecutando `procesar_estados_por_lotes` sobre los estados seleccionados...


llm_error
reintento_programado
llm_error
reintento_programado
llm_error
operacion_fallida
Traceback (most recent call last):
  File "c:\Users\Danny\Documents\Proyecto de CommunityLab MotorInteligente\communitylab\venv\Lib\site-packages\httpx2\_transports\default.py", line 98, in map_httpcore_exceptions
    yield
  File "c:\Users\Danny\Documents\Proyecto de CommunityLab MotorInteligente\communitylab\venv\Lib\site-packages\httpx2\_transports\default.py", line 245, in handle_request
    resp = self._pool.handle_request(req)
  File "c:\Users\Danny\Documents\Proyecto de CommunityLab MotorInteligente\communitylab\venv\Lib\site-packages\httpcore2\_sync\connection_pool.py", line 242, in handle_request
    raise exc from None
  File "c:\Users\Danny\Documents\Proyecto de CommunityLab MotorInteligente\communitylab\venv\Lib\site-packages\httpcore2\_sync\connection_pool.py", line 224, in handle_request
    response = connection.handle_request(pool_request.request)
  File "c:\Users\Danny\Documents\P

✅ Procesamiento finalizado. Total resultados obtenidos: 2

🔹 Resultado ID: int-002
   Sentimiento: neutral | Tema: datos_ia
   Rutas activadas: ['preguntas_frecuentes']
   Activos generados: ['preguntas_frecuentes']

🔹 Resultado ID: int-022
   Sentimiento: muy_positivo | Tema: empleabilidad
   Rutas activadas: ['caso_exito', 'boletin', 'linkedin']
   Activos generados: ['caso_exito', 'boletin', 'linkedin']


---
## 📝 Paso 4: Contrato de Salida y Plantillas de Prompts Few-Shot
Inspeccionamos cómo están estructurados los prompts y el contrato de salida de Ciencia de Datos.

In [14]:
from src.agentes.prompts.linkedin import prompt_linkedin
from src.agentes.prompts.preguntas_frecuentes import prompt_preguntas_frecuentes
from src.agentes.contrato_salida_ciencia_datos import VERSION_CONTRATO_SALIDA_DS

print('=' * 75)
print(f'📋 CONTRATO DE SALIDA DE DATA SCIENCE: Versión {VERSION_CONTRATO_SALIDA_DS}')
print('=' * 75)

print('\n📄 Vista previa de la plantilla Few-Shot para LinkedIn (prompt_linkedin):')
print('-' * 75)
print(prompt_linkedin.messages[0].prompt.template[:420] + '\n... [TRUNCADO PARA VISUALIZACIÓN]')

print('\n❓ Vista previa de la plantilla para FAQs (prompt_preguntas_frecuentes):')
print('-' * 75)
print(prompt_preguntas_frecuentes.messages[0].prompt.template[:400] + '\n... [TRUNCADO PARA VISUALIZACIÓN]')


📋 CONTRATO DE SALIDA DE DATA SCIENCE: Versión 1.3

📄 Vista previa de la plantilla Few-Shot para LinkedIn (prompt_linkedin):
---------------------------------------------------------------------------

Eres redactor del equipo de Marketing y Community Management
de una comunidad educativa y tecnológica.

Tu tarea es transformar un testimonio real de un miembro de la
comunidad en una publicación para el LinkedIn oficial de la organización.

La publicación debe estar escrita desde una voz institucional.
Nunca escribas como si fueras la persona autora del testimonio.

# OBJETIVO

Convertir una experiencia real de la 
... [TRUNCADO PARA VISUALIZACIÓN]

❓ Vista previa de la plantilla para FAQs (prompt_preguntas_frecuentes):
---------------------------------------------------------------------------

Eres redactor de preguntas frecuentes para una comunidad educativa
y tecnológica.

Tu tarea es transformar una pregunta real de la comunidad en una
entrada breve y útil para una sección de pregun

---
## ⚡ Paso 5: Exploración e Integración de NVIDIA NIM (Nemotron)
Siguiendo la arquitectura multimodelo acordada por el equipo de Data Science:
- **Objetivo:** Utilizar **Nemotron (NVIDIA NIM)** como motor principal para evitar el agotamiento de cuota de Gemini, manteniendo a Gemini/Cohere como respaldo (*fallback*).
- **Endpoint compatible con OpenAI:** Se conecta a `https://integrate.api.nvidia.com/v1` mediante `ChatOpenAI` y el modelo `nvidia/nemotron-3.5-lightning-30b-a3b`.
- **Estructuración:** Prepara las instrucciones con el esquema técnico JSON requerido por el contrato de CommunityLab.

In [15]:
from langchain_core.messages import HumanMessage
from src.agentes.proveedores.nvidia_nim import obtener_configuracion_nvidia
from src.agentes.configuracion_ia import MODELO_NVIDIA_NEMOTRON
from src.agentes.modelo_ia import (
    obtener_modelo_analisis_estructurado,
    obtener_modelo_generacion_estructurado,
    invocar_modelo_con_fallback,
)
from src.agentes.modelos import AnalisisMensaje, PublicacionLinkedIn
from src.agentes.fallback_ia import invocar_modelo_con_fallback

config_nv = obtener_configuracion_nvidia()
print("=" * 75)
print("🚀 MÓDULO OFICIAL DE NVIDIA NIM Y ARQUITECTURA DE FALLBACK")
print("=" * 75)
print(f"• Proveedor primario   : {config_nv['proveedor']}")
print(f"• Identificador modelo : {config_nv['identificador_modelo']}")
print(f"• Base URL oficial     : {config_nv['base_url']}")

# 1. Obtener directamente el modelo estructurado de NIM
modelo_nim = obtener_modelo_analisis_estructurado(AnalisisMensaje, proveedor="nvidia_nim")

print("\n🧪 Invocando modelo NIM estructurado directamente...")
# El adaptador de NIM recibe una lista de mensajes de LangChain (HumanMessage) o un prompt formateado
mensajes_entrada = [
    HumanMessage(content="Excelente curso de Machine Learning, me ayudó a conseguir trabajo como Data Scientist!")
]
resultado = modelo_nim.invoke(mensajes_entrada)
print(f"   ✅ Sentimiento    : {resultado.sentimiento}")
print(f"   ✅ Tema principal : {resultado.tema_principal}")
print(f"   ✅ Subtema        : {resultado.subtema}")
print(f"   ✅ Tipo detectado : {resultado.tipo_detectado}")
print("\n🎉 ¡NVIDIA NIM responde bajo el contrato oficial de arquitectura del proyecto!")


🚀 MÓDULO OFICIAL DE NVIDIA NIM Y ARQUITECTURA DE FALLBACK
• Proveedor primario   : NVIDIA NIM
• Identificador modelo : nvidia/nemotron-3.5-lightning-30b-a3b
• Base URL oficial     : https://integrate.api.nvidia.com/v1

🧪 Invocando modelo NIM estructurado directamente...


TypeError: Formato de entrada no compatible con NVIDIA NIM

---
## 🎯 Resumen de Estado del Equipo y Próximos Aportes

### 📍 Estado Actual del Proyecto y Sincronización con el Equipo:
- **Línea base integrada:** `develop` tiene consolidada la suite de Data Science (PR #22) con 152 tests en verde.
- **Transición a Arquitectura Multi-LLM (Nemotron + Fallback):**
  - El compañero **Arnold Bustinza (DS)** lidera la rama `DS-Semana3-Nvidia` para integrar Nemotron (NVIDIA NIM) como LLM principal y mitigar los límites de cuota de Gemini.
  - El compañero **Gustavo Vásquez (DA)** reportó el fix de retorno de cadena en `src/agentes/proveedores/nvidia_nim.py` y el cuidado con el hardening en `nodo_analizador.py`.
  - Nosotros ya dejamos validado y operativo Nemotron con fallback a Cohere en nuestro entorno local y notebook.

### 💡 Próximos Aportes de Alto Impacto para Danny (Data Scientist):
1. **Revisión y Aprobación de PR (Code Review a Arnold):**
   - En cuanto Arnold suba la corrección del adaptador Nvidia a su rama remota, descargarla, ejecutar los tests de integración (`tests/agents/test_adaptador_nvidia_integracion.py`) y validar la resolución de conflictos con `develop`.
2. **Benchmarking de Calidad Multi-Modelo (Nemotron vs Cohere vs DeepSeek):**
   - Evaluar la consistencia semántica en los casos límite (`int-006`, `int-011`, `int-016`, `int-020`) comparando la precisión de clasificación y JSON Schema entre proveedores.
3. **Revisión Cruzada de Contratos de Salida (PR #24 y PR #25):**
   - Verificar compatibilidad de salida v1.3 con el filtro de ruido de DA (Jhonattan - PR #24) y los esquemas JSON de Arquitectura (Ramses - PR #25).
4. **Retroalimentación y Calibración:**
   - Apoyar en la conexión entre la interfaz de curaduría de Streamlit y las métricas de aprobación humana para retroalimentar el pipeline.